# Slingshot Solver 2.0
## Gravitational Slingshot Dynamics in Restricted 3-Body Systems

Refactored modular implementation with:
- Unified Monte Carlo workflow (barycentric & planet-frame)
- Configuration-driven pipeline (YAML/JSON)
- Robust trajectory analysis and encounter extraction
- Animation/video rendering support
- Parallelized particle evaluation

**Core paper**: [Add reference]

**System**: Kepler-432 (K0V star, 1.19 M☉; hot Jupiter, 5.2 M♃, a≈0.0896 AU)

## Setup and Configuration

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add package to path
sys.path.insert(0, '/path/to/slingshot-solver')  # Adjust path as needed

import slingshot
from slingshot.config import load_config, FullConfig
from slingshot.monte_carlo import run_monte_carlo, select_top_indices
from slingshot.plotting import plot_mc_summary, plot_best_candidate_with_bodies
from slingshot.animation import generate_all_animations
from slingshot.baselines import compare_3body_with_baselines

print(f"Slingshot Solver v{slingshot.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")

### Load Configuration

Choose between:
1. **Load from file** (recommended): YAML or JSON config
2. **Create in-place**: Python dict or FullConfig object
3. **Predefined system**: Quick setup by system name

In [ ]:
# Option 1: Load from YAML file
# cfg = load_config('config_kepler432.yaml')

# Option 2: Create configuration in Python
from slingshot.config import (
    SystemConfig, SamplingConfig, NumericalConfig,
    PipelineConfig, VisualizationConfig, FullConfig
)

cfg = FullConfig(
    system=SystemConfig(
        name="Kepler-432",
        M_star_Msun=1.19,
        M_planet_Mjup=5.2,
        R_planet_Rjup=1.155,
        a_planet_AU=0.0896,
    ),
    sampling=SamplingConfig(
        mode="barycentric",
        v_mag_min_kms=10.0,
        v_mag_max_kms=120.0,
        impact_param_min_AU=0.5,
        impact_param_max_AU=3.0,
        bary_unbound_requirement="both",
    ),
    numerical=NumericalConfig(
        rtol=1e-10,
        atol=1e-10,
        r_far_factor=20.0,
    ),
    pipeline=PipelineConfig(
        N_particles=3000,
        t_mc_max_sec=1e7,
        t_best_max_sec=1e7,
        top_frac=0.10,
        select_metric="bary_delta_v_pct",
        n_parallel=None,  # None = serial, >1 = parallel
    ),
    visualization=VisualizationConfig(
        render_video=True,
        video_fps=30,
        animate_trajectory=True,
        animate_phase_space=True,
    ),
)

print("Configuration loaded:")
print(f"  System: {cfg.system.name}")
  Particles: {cfg.pipeline.N_particles}")
print(f"  Sampling: {cfg.sampling.mode}")
print(f"  Parallelization: {cfg.pipeline.n_parallel or 'Serial'}")

## Phase 1: Monte Carlo Sweep

Sample and integrate N test particles through encounters.

In [ ]:
# Convert config to physical units
M_SUN = 1.98847e30  # kg
M_JUP = 1.898e27  # kg
R_JUP = 71492.0  # km
AU_KM = 1.495978707e8  # km

m_star = cfg.system.M_star_Msun * M_SUN
m_p = cfg.system.M_planet_Mjup * M_JUP
R_p = cfg.system.R_planet_Rjup * R_JUP

print(f"Physical parameters:")
print(f"  Star: {cfg.system.M_star_Msun:.2f} M☉")
print(f"  Planet: {cfg.system.M_planet_Mjup:.1f} M♃ (R={R_p:.0f} km)")
print(f"  Orbit: {cfg.system.a_planet_AU:.4f} AU")
print()

# Run Monte Carlo
print("Running Monte Carlo sweep...")
print(f"  Particles: {cfg.pipeline.N_particles}")
print(f"  Timespan: {cfg.pipeline.t_mc_max_sec:.1e} s")

mc = run_monte_carlo(
    N=cfg.pipeline.N_particles,
    t_span=(0.0, cfg.pipeline.t_mc_max_sec),
    m_star=m_star,
    m_p=m_p,
    R_p=R_p,
    frame="barycentric",
    sampling_mode=cfg.sampling.mode,
    n_parallel=cfg.pipeline.n_parallel,
    verbose=True,
    # Pass sampling params
    v_mag_min=cfg.sampling.v_mag_min_kms,
    v_mag_max=cfg.sampling.v_mag_max_kms,
    impact_param_min_AU=cfg.sampling.impact_param_min_AU,
    impact_param_max_AU=cfg.sampling.impact_param_max_AU,
    angle_in_min_deg=cfg.sampling.angle_in_min_deg,
    angle_in_max_deg=cfg.sampling.angle_in_max_deg,
    # Numerical params
    rtol=cfg.numerical.rtol,
    atol=cfg.numerical.atol,
    r_far_factor=cfg.numerical.r_far_factor,
    min_clearance_factor=cfg.numerical.min_clearance_factor,
    # Filtering
    bary_unbound_requirement=cfg.sampling.bary_unbound_requirement,
)

ok_count = mc["ok"].sum()
print(f"\nResults: {ok_count}/{cfg.pipeline.N_particles} successful ({100.0*ok_count/cfg.pipeline.N_particles:.1f}%)")

### MC Summary Visualization

In [ ]:
fig = plot_mc_summary(mc, figsize=(12, 5))
plt.show()

# Print statistics
delta_v_success = mc["delta_v"][mc["ok"]]
deflection_success = mc["deflection"][mc["ok"]]

print(f"\nMonte Carlo Summary Statistics:")
print(f"  Δv range: [{delta_v_success.min():.2f}, {delta_v_success.max():.2f}] km/s")
print(f"  Δv mean: {delta_v_success.mean():.2f} ± {delta_v_success.std():.2f} km/s")
print(f"  Deflection range: [{deflection_success.min():.1f}, {deflection_success.max():.1f}]°")
print(f"  Deflection mean: {deflection_success.mean():.1f} ± {deflection_success.std():.1f}°")

## Phase 2: Top Candidate Selection

In [ ]:
top_idx = select_top_indices(
    mc,
    top_frac=cfg.pipeline.top_frac,
    min_top=cfg.pipeline.min_top,
    metric="delta_v",
    sign=cfg.pipeline.select_sign,
)

print(f"Selected {len(top_idx)} top candidates (top {cfg.pipeline.top_frac*100:.0f}%)")
print(f"Indices: {top_idx}")

# Show top candidates
print("\nTop 5 candidates:")
for rank, idx in enumerate(top_idx[:5], 1):
    dv = mc["delta_v"][idx]
    defl = mc["deflection"][idx]
    print(f"  {rank}. MC#{idx}: Δv={dv:.2f} km/s, deflection={defl:.1f}°")

## Phase 3: High-Resolution Re-run

Re-integrate top candidates at higher resolution.

In [ ]:
from slingshot.dynamics import simulate_3body, init_hot_jupiter_barycentric
from slingshot.analysis import analyze_trajectory

print(f"Re-running {len(top_idx)} candidates at high resolution...")
print(f"  Timespan: {cfg.pipeline.t_best_max_sec:.1e} s")
print(f"  Output points: {cfg.pipeline.n_eval_best}")

Y_sp0 = mc["Y_sp0"]
sat_states = mc["sat_states"]

sols_best = []
analyses_best = []
Y0_best = []

for i, idx in enumerate(top_idx):
    if i % max(1, len(top_idx)//10) == 0:
        print(f"  Progress: {i+1}/{len(top_idx)}")
    
    # Reconstruct initial state
    xs, ys, vxs, vys, xp, yp, vxp, vyp = Y_sp0
    x3, y3, vx3, vy3 = sat_states[idx]
    Y0 = np.array([xs, ys, vxs, vys, xp, yp, vxp, vyp, x3, y3, vx3, vy3], dtype=float)
    
    # Integrate at high resolution
    sol = simulate_3body(
        Y0,
        (0.0, cfg.pipeline.t_best_max_sec),
        m_star=m_star,
        m_p=m_p,
        n_eval=cfg.pipeline.n_eval_best,
        rtol=cfg.numerical.rtol,
        atol=cfg.numerical.atol,
    )
    
    # Analyze
    ana = analyze_trajectory(
        sol,
        frame="barycentric",
        m_star=m_star,
        m_p=m_p,
        R_p=R_p,
        r_far_factor=cfg.numerical.r_far_factor,
        min_clearance_factor=cfg.numerical.min_clearance_factor,
    )
    
    sols_best.append(sol)
    analyses_best.append(ana)
    Y0_best.append(Y0)

print("Re-run complete.")

## Phase 4: Detailed Analysis & Visualization

### Best Candidate Trajectory

In [ ]:
# Find best by Δv
best_local_idx = max(
    range(len(analyses_best)),
    key=lambda i: analyses_best[i]["delta_v"] if analyses_best[i] else -np.inf
)

best_sol = sols_best[best_local_idx]
best_ana = analyses_best[best_local_idx]
best_original_idx = top_idx[best_local_idx]

print("="*60)
print(f"BEST CANDIDATE (MC index #{best_original_idx})")
print("="*60)
print(f"Δv:              {best_ana['delta_v']:>12.2f} km/s")
print(f"Δv %:            {best_ana['delta_v_pct']:>12.1f} %")
print(f"Deflection:      {best_ana['deflection']:>12.1f}°")
print(f"Deflection/180:  {best_ana['deflection_frac']:>12.3f}")
print(f"r_min:           {best_ana['r_min']:>12.0f} km")
print(f"Impact param:    {best_ana['impact_parameter']:>12.0f} km")
print(f"Unbound (final): {str(best_ana['unbound_f']):>12s}")
print(f"Eps_i:           {best_ana['eps_i']:>12.3e} km²/s²")
print(f"Eps_f:           {best_ana['eps_f']:>12.3e} km²/s²")
print("="*60)

In [ ]:
fig = plot_best_candidate_with_bodies(best_sol, best_ana, m_star=m_star, m_p=m_p, R_p=R_p)
plt.show()

### Phase-Space Analysis

In [ ]:
from slingshot.plotting import plot_velocity_phase_space

fig = plot_velocity_phase_space(best_sol, title_prefix="Best Candidate", figsize=(12, 5))
plt.show()

### Baseline Comparison

Compare with 2-body hyperbola and monopole baselines.

In [ ]:
from slingshot.analysis import extract_encounter_states

# Extract encounter geometry
enc = extract_encounter_states(
    best_sol,
    m_p=m_p,
    R_p=R_p,
    r_far_factor=cfg.numerical.r_far_factor,
    min_clearance_factor=cfg.numerical.min_clearance_factor,
)

if enc.ok:
    # Compare with baselines
    comparison = compare_3body_with_baselines(
        best_sol,
        enc,
        m_star=m_star,
        m_p=m_p,
        R_p=R_p,
        make_plots=True,
    )
    
    if comparison["ok"]:
        print(f"\nBaseline Comparison Results:")
        print(f"  2-body deflection: {comparison['two_body']['deflection_deg']:.1f}°")
        print(f"  3-body deflection: {comparison.get('delta_3body', np.nan):.1f}°")
        print(f"  Extra Δv from planet: {comparison['extra_eps_from_planet']:.3e} km²/s²")
else:
    print(f"Encounter extraction failed: {enc.reason}")

### Animation/Video Generation

Create animation frames for visual analysis.

In [ ]:
if cfg.visualization.render_video:
    print(f"Generating animations...")
    print(f"  Output dir: ./frames")
    print(f"  Format: {cfg.visualization.video_format}")
    print(f"  FPS: {cfg.visualization.video_fps}")
    
    animations = generate_all_animations(
        best_sol,
        output_dir="./frames",
        video_fps=cfg.visualization.video_fps,
        video_format=cfg.visualization.video_format,
        animate_trajectory=cfg.visualization.animate_trajectory,
        animate_phase_space=cfg.visualization.animate_phase_space,
    )
    
    for anim_type, filepath in animations.items():
        if filepath:
            print(f"  ✓ {anim_type}: {filepath}")
        else:
            print(f"  ✗ {anim_type}: Failed")
else:
    print("Video rendering disabled in config.")

## Exploratory Analysis: All Candidates

In [ ]:
# Create comparison table of all re-run candidates
candidates_table = []
for i, (idx, sol, ana) in enumerate(zip(top_idx, sols_best, analyses_best)):
    if ana:
        candidates_table.append({
            'Rank': i+1,
            'MC_idx': idx,
            'Δv (km/s)': f"{ana['delta_v']:.2f}",
            'Δv (%)': f"{ana['delta_v_pct']:.1f}",
            'Deflection (°)': f"{ana['deflection']:.1f}",
            'r_min (km)': f"{ana['r_min']:.0f}",
        })

import pandas as pd
df = pd.DataFrame(candidates_table)
print("\nTop Candidates Summary:")
print(df.to_string(index=False))

## Save Configuration & Results

Store configuration and results for reproducibility.

In [ ]:
from slingshot.config import save_config
import pickle
from datetime import datetime

output_dir = Path(f"./results_{cfg.system.name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
output_dir.mkdir(parents=True, exist_ok=True)

# Save config
config_file = output_dir / "config.yaml"
save_config(cfg, str(config_file), format="yaml")
print(f"✓ Config saved: {config_file}")

# Save results
results_file = output_dir / "results.pkl"
with open(results_file, 'wb') as f:
    pickle.dump({
        'mc': mc,
        'top_indices': top_idx,
        'sols_best': sols_best,
        'analyses_best': analyses_best,
    }, f)
print(f"✓ Results saved: {results_file}")

# Save summary table
summary_file = output_dir / "summary.csv"
df.to_csv(summary_file, index=False)
print(f"✓ Summary saved: {summary_file}")

print(f"\nAll outputs in: {output_dir}")

## Next Steps & Customization

### Experiment Variations

- **Different systems**: Change `cfg.system` to TOI-1431 or custom values
- **Sampling modes**: Switch `cfg.sampling.mode` from 'barycentric' to 'planet'
- **Selection metrics**: Try 'bary_delta_v', 'deflection', etc.
- **Parallelization**: Set `cfg.pipeline.n_parallel` to >1

### Production Workflow

1. **Save config**: `save_config(cfg, 'my_experiment.yaml')`
2. **Run from config**: `cfg = load_config('my_experiment.yaml')`
3. **Reproducible**: Exact same configuration and results

### Debugging & Diagnostics

- Access individual particle results: `mc['results'][idx]`
- Detailed encounter geometry: `best_ana['encounter']`
- Check why particles failed: `[r for r in mc['results'] if not r['ok']][:5]`